In [0]:
# Install all required packages
%pip install clickhouse-connect langchain langchain-community langchain-openai python-dotenv

In [0]:
# Restart Python to use newly installed packages
dbutils.library.restartPython()

In [0]:
import os
from dotenv import load_dotenv
import clickhouse_connect

# Load environment variables from .env file
load_dotenv()

# Connect to ClickHouse
try:
    client = clickhouse_connect.get_client(
        host=os.getenv('CH_HOST'),
        port=8443,
        username='default',
        password=os.getenv('CH_PASSWORD'),
        secure=True
    )
    print(f"✅ Connection successful! ClickHouse version: {client.server_version}")
except Exception as e:
    print(f"❌ Connection failed. Did you create the .env file? Error: {e}")

In [0]:
# Configure LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Academic-ClickHouse-Project"
os.environ["LANGCHAIN_API_KEY"] = os.getenv('LANGCHAIN_API_KEY')

print("📡 LangSmith tracing is now ACTIVE.")

In [0]:
# Create table to store AI logs
client.command("""
CREATE TABLE IF NOT EXISTS ai_logs (
    event_time DateTime DEFAULT now(),
    prompt String,
    response String,
    tokens_used UInt32
) ENGINE = MergeTree()
ORDER BY event_time
""")

print("✅ Table 'ai_logs' is ready in ClickHouse!")

In [0]:
from langchain_community.llms.fake import FakeListLLM
from langchain_core.prompts import PromptTemplate

# Setup Fake LLM with predefined responses
responses = ["The capital of ClickHouse is Speed!", "Data is the new electricity."]
llm = FakeListLLM(responses=responses)

# Create prompt template
prompt = PromptTemplate(
    input_variables=["topic"],
    template="Tell me something about {topic}."
)

# Build and run chain using LCEL
topic = "Database Performance"
chain = prompt | llm
ai_response = chain.invoke({"topic": topic})

print(f"🤖 AI says: {ai_response}")

# Save result to ClickHouse
client.command(f"""
INSERT INTO ai_logs (prompt, response, tokens_used) VALUES 
('{topic}', '{ai_response}', 42)
""")

print("✅ Data successfully pushed to ClickHouse & Traced in LangSmith!")

In [0]:
# Query ClickHouse to view logged AI interactions
results = client.query("SELECT event_time, prompt, response FROM ai_logs ORDER BY event_time DESC LIMIT 5")

# Format and display results
print(f"{'TIME':<20} | {'PROMPT':<20} | {'RESPONSE'}")
print("-" * 70)
for row in results.result_rows:
    print(f"{str(row[0]):<20} | {row[1]:<20} | {row[2]}")